# 4-5 子图

当需要同时比较多个图表时，可以使用 `plt.subplots()` 在一张图中创建多个绘图区。

| 对象 | 含义 |
|---|---|
| `Figure` | 整张画布，可以包含多个绘图区 |
| `Axes` | 一个具体的绘图区，具有自己的标题、坐标轴和图形 |

> `Axes` 表示一个绘图区，不要与表示单条坐标轴的 `Axis` 混淆。

> 约定：使用 `np` 作为 NumPy 的别名，使用 `plt` 作为 `matplotlib.pyplot` 的别名。
> 下方准备单元格会选择可用的中文字体；可以从项目根目录或本章目录运行。
> `fonts` 文件夹是可选的，没有该文件夹时会查找系统字体。
> 如果没有可用的中文字体，会显示提示并使用默认字体继续绘图，中文可能显示为方框。
> 字体设置只作用于当前 Python 进程，详细原理见第 4-6 节。

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager

# fonts 文件夹是可选的；没有字体文件时，使用系统已安装的字体。
project_dir = Path('.')
if Path.cwd().name == '第4章-Matplotlib':
    project_dir = Path('..')
font_files = [
    project_dir / 'fonts' / 'NotoSansCJKsc-Regular.otf',
    project_dir / 'fonts' / 'NotoSansSC-Regular.ttf',
]
font_path = next((path for path in font_files if path.is_file()), None)

if font_path is None:
    font_candidates = [
        'Microsoft YaHei', 'SimHei', 'Noto Sans CJK SC', 'Noto Sans SC',
        'Source Han Sans SC', 'PingFang SC', 'Heiti SC', 'Songti SC',
        'Arial Unicode MS',
    ]
    available_fonts = {entry.name for entry in font_manager.fontManager.ttflist}
    font_name = next((name for name in font_candidates if name in available_fonts), None)
    if font_name is None:
        font_name = 'DejaVu Sans'
        print('未找到中文字体，暂用 DejaVu Sans，图中的中文可能显示为方框。')
        print('可安装 Noto Sans CJK SC，或将中文字体文件放入项目的 fonts 文件夹后重新运行。')
    font_path = Path(font_manager.findfont(
        font_manager.FontProperties(family=font_name), fallback_to_default=False
    ))
else:
    font_name = font_manager.FontProperties(fname=str(font_path)).get_name()

font_manager.fontManager.addfont(str(font_path))
plt.rcParams['font.family'] = [font_name]
plt.rcParams['axes.unicode_minus'] = False

## 1. 准备数据

下面准备两组销量、四类数量，以及 500 个固定种子的随机样本。

In [ ]:
x_data = [2011, 2012, 2013, 2014, 2015, 2016, 2017]
y_data = [58000, 60200, 63000, 71000, 84000, 90500, 107000]
y_data2 = [52000, 54200, 51500, 58300, 56800, 59500, 62700]
bar_x = ['a', 'b', 'c', 'd']
bar_y = [20, 10, 30, 25]
rng = np.random.default_rng(42)
samples = rng.standard_normal(500)

## 2. 创建一行两列子图

```python
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(10, 4))
```

- `nrows`：子图行数。
- `ncols`：子图列数。
- `figsize`：整张画布的宽和高，单位是英寸，不是每个子图的尺寸。
- 一行两列时，`axes[0]` 和 `axes[1]` 分别表示左、右两个绘图区。

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(10, 4))
axes[0].plot(x_data, y_data, marker='o')
axes[1].plot(x_data, y_data2, color='orange', marker='o')
axes[0].set_title('产品 A 销量')
axes[1].set_title('产品 B 销量')

for ax in axes:
    ax.set_xlabel('年份')
    ax.set_ylabel('销量')
    ax.set_xticks([2011, 2013, 2015, 2017])
    ax.grid(axis='y', alpha=0.3)

fig.tight_layout()
plt.show()

上面两个子图默认各自确定纵轴范围，因此不能只根据曲线的视觉高度比较两组销量。
第 4 部分会使用共享纵轴统一尺度。

## 3. 创建两行两列子图

多行多列时，`axes` 默认是二维数组，用 `axes[行索引, 列索引]` 选择绘图区，索引从 `0` 开始。

| 索引 | 位置 |
|---|---|
| `axes[0, 0]` | 左上 |
| `axes[0, 1]` | 右上 |
| `axes[1, 0]` | 左下 |
| `axes[1, 1]` | 右下 |

每个绘图区可以使用不同的图表类型。

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(10, 7))

axes[0, 0].plot(x_data, y_data, marker='o')
axes[0, 0].set(title='产品 A 销量', xlabel='年份', ylabel='销量')
axes[0, 0].set_xticks([2011, 2013, 2015, 2017])

axes[0, 1].plot(x_data, y_data2, color='orange', marker='o')
axes[0, 1].set(title='产品 B 销量', xlabel='年份', ylabel='销量')
axes[0, 1].set_xticks([2011, 2013, 2015, 2017])

axes[1, 0].bar(bar_x, bar_y, color='seagreen')
axes[1, 0].set(title='各类别数量', xlabel='类别', ylabel='数量')

axes[1, 1].hist(samples, bins=15, rwidth=0.8, color='steelblue')
axes[1, 1].set(title='随机样本分布', xlabel='样本值', ylabel='频数')

fig.suptitle('两行两列子图示例')
fig.tight_layout()
plt.show()

`ax.set(title=..., xlabel=..., ylabel=...)` 可以同时设置多个属性，
与分别调用 `set_title()`、`set_xlabel()`、`set_ylabel()` 的效果相同。
`fig.tight_layout()` 自动调整间距，减少标题与刻度标签重叠。

## 4. 共享坐标轴

| 参数 | 作用 |
|---|---|
| `sharex=True` | 子图共享横轴范围与刻度设置 |
| `sharey=True` | 子图共享纵轴范围与刻度设置 |

数据单位和含义相同时，共享坐标轴有助于公平比较。不同含义的图表不宜强行共享坐标轴。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
axes[0].plot(x_data, y_data, marker='o')
axes[1].plot(x_data, y_data2, marker='o', color='orange')
axes[0].set_title('产品 A')
axes[1].set_title('产品 B')
axes[0].set_ylabel('销量')
axes[0].set_ylim(0, 120000)

for ax in axes:
    ax.set_xlabel('年份')
    ax.set_xticks([2011, 2013, 2015, 2017])
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('共享横轴与纵轴的销量对比')
fig.tight_layout()
plt.show()

现在两个子图使用相同纵轴范围。对其中一个共享纵轴调用 `set_ylim()`，另一个也会更新。
共享坐标轴时，部分重复刻度标签可能被自动隐藏。

## 5. 理解 `axes` 的形状

默认 `squeeze=True` 会压缩长度为 1 的维度：

| 子图布局 | `axes` 的结果 | 访问方式 |
|---|---|---|
| `1 × 1` | 单个 `Axes` 对象 | `ax.plot(...)` |
| `1 × n` 或 `n × 1` | 一维数组 | `axes[0].plot(...)` |
| `m × n`，且 `m, n > 1` | 二维数组 | `axes[0, 0].plot(...)` |

如果希望统一使用二维索引，可以指定 `squeeze=False`。

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(6, 4), squeeze=False)
print('axes 的形状：', axes.shape)
axes[0, 0].bar(bar_x, bar_y)
axes[0, 0].set(title='使用二维索引访问单个子图', xlabel='类别', ylabel='数量')
fig.tight_layout()
plt.show()

## 6. 要点总结

- `plt.subplots()` 同时返回画布 `fig` 和绘图区 `axes`。
- `nrows`、`ncols` 控制布局，`figsize` 控制整张画布尺寸。
- 一维布局通常使用 `axes[i]`，二维布局使用 `axes[i, j]`。
- 各子图调用自己的 `plot()`、`bar()`、`hist()` 和标题设置方法。
- `sharex`、`sharey` 适合同单位、同含义数据的比较。
- `tight_layout()` 调整间距，`squeeze=False` 可以统一二维访问方式。